# EEG Frequency Bands

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1 (single subject for speed)

---

## Overview

EEG signals are classified into five main frequency bands, each associated with a different brain state. This notebook applies spectral analysis (FFT) to a real EEG signal and highlights the bands present in it.

| Band | Frequency | Brain state |
| ------- | ------- | --------- |
| Delta | 0.5 to 4 Hz | Deep sleep |
| Theta | 4 to 8 Hz | Drowsiness, memory |
| Alpha | 8 to 13 Hz | Relaxation (eyes closed) |
| Beta | 13 to 30 Hz | Active thinking, focus |
| Gamma | 30 to 100 Hz | Higher cognitive processing |

## Expected output

A plot showing power distribution across frequencies from 0 to 100 Hz. Power concentrates in low bands (delta, theta, alpha) and decreases at higher frequencies, following the $1/f$ spectral law.

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone repository and download one subject

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

## 3. Load EEG signal

We load subject 1, experiment 1, session 2, channel P4 (parietal region).

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

## 4. Compute power spectral density

We use Welch's method to estimate the power spectral density. This method splits the signal into overlapping segments, computes the spectrum for each, and averages them for a smoother estimate.

In [ ]:
from scipy import signal

freqs, psd = signal.welch(channel_data, fs=fs, nperseg=1024)
print(f'Frequency range: {freqs[0]:.1f} to {freqs[-1]:.1f} Hz')
print(f'Frequency resolution: {freqs[1]-freqs[0]:.2f} Hz')

## 5. Interactive plot of frequency bands

**What to look for:**
- Power concentrates in low bands (delta, theta, alpha)
- Power decreases with frequency (1/f spectral law)
- A peak may appear in the alpha band around 10 Hz if the subject was relaxed
- A peak at 50 Hz may appear due to power line interference

In [ ]:
import plotly.graph_objects as go

bands = {
    'Delta (0.5-4 Hz)':   (0.5, 4, 'purple'),
    'Theta (4-8 Hz)':     (4, 8, 'blue'),
    'Alpha (8-13 Hz)':    (8, 13, 'green'),
    'Beta (13-30 Hz)':    (13, 30, 'orange'),
    'Gamma (30-100 Hz)':  (30, 100, 'red'),
}

fig = go.Figure()
fig.add_trace(go.Scatter(x=freqs, y=psd, mode='lines',
                         name='PSD', line=dict(color='black', width=1)))

for name, (lo, hi, color) in bands.items():
    mask = (freqs >= lo) & (freqs <= hi)
    fig.add_trace(go.Scatter(x=freqs[mask], y=psd[mask], mode='lines',
                             name=name, line=dict(color=color, width=2),
                             fill='tozeroy', opacity=0.3))

fig.update_layout(height=500, title='EEG Frequency Bands (P4, subject 1)',
                  xaxis_title='Frequency (Hz)', yaxis_title='Power (uV^2/Hz)',
                  yaxis_type='log', xaxis_range=[0, 100])
fig.show()

## 6. Summary

- EEG contains five main frequency bands, each linked to a brain state
- Power concentrates in low bands and decreases with frequency (1/f law)
- Welch's method gives a smoother spectral estimate than direct FFT
- This analysis is the first step to understanding what information the signal carries before applying filters in Chapter 4